# Session 9 - CSET training


**GPU T4, ~5 h.** Causally-Supervised Explanation Tuning (CSET).

Three steps: audit the base model on **DEV** to measure Delta, build two label sets,
and QLoRA-tune one adapter per label set.

* `causal` -- the citation target is argmax Delta (the region that moves the verdict)
* `mask` -- the citation target is the region where the pixels changed (the standard supervision)

The verdict target is the true label in both, so detection is trained identically and
only the citation supervision differs. **DEV only** -- the module refuses to build a
training set containing TEST rows.

In [ ]:
SESSION = "S9 CSET"

# ============================== CONFIG ==============================
BASE_MODEL  = "Qwen/Qwen2.5-VL-3B-Instruct"
DETECTOR    = "qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct"
QUANT       = "fp16"
MAX_PIXELS  = 384 * 384
SEED        = 0

DEV_SAMPLES     = 3000     # DEV audit size; 0 = all DEV
DEV_BUDGET_MIN  = 150

VARIANTS    = ["causal", "mask"]
MIN_DELTA   = 0.02         # drop DEV samples with no causal signal
LORA_R      = 16
LR          = 1e-4
EPOCHS      = 1
BATCH_SIZE  = 4
GRAD_ACCUM  = 4
TRAIN_BUDGET_MIN = 150     # per variant -> 150 + 2x150 = 7.5 h total

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils peft bitsandbytes")
KU.gpu_report()
INDEX = KU.find_parsed_index()
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")
n_gpu = max(1, KU.n_gpus())
print("DEV samples in index:", len(C.filter_split(recs, "dev")))

In [ ]:
# ==================== 1. AUDIT THE BASE MODEL ON DEV ====================
RESUME = ",".join(d for d in KU.find_run_dirs() if "run_dev" in d)
cmds, envs, logs = [], [], []
for i in range(n_gpu):
    logs.append(f"{OUT}/logs/dev_{i}.log")
    envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
    cmds.append(
        f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
        f'--detector "{DETECTOR}" --out "{OUT}/run_dev" --tag dev '
        f'--split dev --limit-samples {DEV_SAMPLES} --seed {SEED} '
        f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
        f'--max-pixels {MAX_PIXELS} --vocab {VOCAB} '
        f'--time-budget-min {DEV_BUDGET_MIN}'
        + (f' --resume-from "{RESUME}"' if RESUME else ""))
KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

# Ground truth for the `mask` variant, on DEV.
KU.sh(f'{sys.executable} -m ccaudit.m10_localization --index "{INDEX}" '
      f'--out "{OUT}/loc_dev" --split dev', check=False)

In [ ]:
# ==================== 2. INSPECT THE TRAINING SETS BEFORE TRAINING ====================
# --dry-run is inexpensive and detects an empty or lopsided label set before
# training time is spent on it.
for v in VARIANTS:
    loc = f' --localization "{OUT}/loc_dev/localization.json"' if v == "mask" else ""
    KU.sh(f'{sys.executable} -m ccaudit.m13_cset --raw "{OUT}/run_dev" '
          f'--index "{INDEX}" --out "{OUT}/cset" --variant {v} '
          f'--min-delta {MIN_DELTA} --vocab {VOCAB}{loc} --dry-run',
          check=False)

In [ ]:
# ==================== 3. QLoRA TRAINING ====================
for v in VARIANTS:
    loc = f' --localization "{OUT}/loc_dev/localization.json"' if v == "mask" else ""
    print(C.banner(f"CSET training: {v}"))
    KU.sh(f'{sys.executable} -m ccaudit.m13_cset --raw "{OUT}/run_dev" '
          f'--index "{INDEX}" --out "{OUT}/cset" --variant {v} '
          f'--base-model "{BASE_MODEL}" --min-delta {MIN_DELTA} '
          f'--lora-r {LORA_R} --lr {LR} --epochs {EPOCHS} '
          f'--batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} '
          f'--max-pixels {MAX_PIXELS} --seed {SEED} --vocab {VOCAB}{loc} '
          f'--time-budget-min {TRAIN_BUDGET_MIN}',
          check=False, log=f"{OUT}/logs/cset_{v}.log")
    log = C.load_json(f"{OUT}/cset/cset_{v}_log.json", {})
    print(f"  final loss {log.get('final_loss')}  steps {log.get('steps')}  "
          f"adapter {log.get('adapter_dir')}")

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s9-cset` (adapters are ~50 MB each).
2. Continue with Sessions 10a and 10b to audit base / cset-causal / cset-mask
   on TEST.
3. The two variants differ only in citation supervision, so the comparison
   isolates the effect of the supervision target; both outcomes are reported
   as measured, without re-tuning."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)